# Diebold-Marino Test

In [ ]:
import numpy as np
from scipy.stats import norm

def dm_test(errors_model1, errors_model2, h=1, lag=None):
    """
    Diebold-Mariano test for equal predictive accuracy (squared-error loss).

    Parameters
    ----------
    errors_model1 : array-like
        Forecast errors of model 1 (y - y_hat1), length N.
    errors_model2 : array-like
        Forecast errors of model 2 (y - y_hat2), length N.
    h : int, optional
        Forecast horizon (in periods). Used only if lag is None.
    lag : int, optional
        Newey-West truncation lag for HAC variance.
        If None, lag is set to max(h-1, 0).

    Returns
    -------
    dm_stat : float
        Diebold-Mariano test statistic.
    p_value : float
        Two-sided p-value under asymptotic N(0,1).
    """

    e1 = np.asarray(errors_model1)
    e2 = np.asarray(errors_model2)

    # Drop NaNs in parallel
    mask = np.isfinite(e1) & np.isfinite(e2)
    e1 = e1[mask]
    e2 = e2[mask]

    if e1.shape != e2.shape:
        raise ValueError("Error series must have the same length after NaN removal.")

    N = len(e1)
    if N < 5:
        raise ValueError("Not enough observations for DM test.")

    # Squared-error loss
    L1 = e1 ** 2
    L2 = e2 ** 2

    # Loss differential
    d = L1 - L2
    d_bar = np.mean(d)

    # Newey–West HAC variance of d_t
    if lag is None:
        lag = max(h - 1, 0)

    d_centered = d - d_bar
    gamma0 = np.dot(d_centered, d_centered) / N
    var_d = gamma0

    for k in range(1, lag + 1):
        cov = np.dot(d_centered[k:], d_centered[:-k]) / N
        weight = 1.0 - k / (lag + 1)  # Bartlett weight
        var_d += 2.0 * weight * cov

    dm_stat = d_bar / np.sqrt(var_d / N)
    p_value = 2 * (1 - norm.cdf(np.abs(dm_stat)))

    return dm_stat, p_value


In [ ]:
# -----------------------------
# 1) Collect prediction dicts
# -----------------------------
model_preds = {
    "RW":    rw_pred_series,      # dict[(maturity, h)] -> forecast series
    "DNS":   dns_pred_series,
    "Ridge": ridge_pred_series,
    "XGB":   xgb_pred_series,
}

# Corresponding actual series (they *should* all be identical per (mat, h),
# but we will just use RW's as canonical ground truth)
model_actuals = {
    "RW":    rw_actual_series,
    "DNS":   dns_actual_series,
    "Ridge": ridge_actual_series,
    "XGB":   xgb_actual_series,
}

maturity_cols = ["DGS2", "DGS5", "DGS10"]
horizons = [1, 5, 10, 30]

dm_results = []

for maturity in maturity_cols:
    for h in horizons:
        key = (maturity, h)

        # Use RW's actual series as canonical ground truth for this (maturity, h)
        if key not in rw_actual_series:
            continue

        actual = rw_actual_series[key].copy()
        actual.name = "actual"

        # Build a dict of aligned prediction series for all models that exist
        preds_for_key = {}
        for model_name, pred_dict in model_preds.items():
            if key not in pred_dict:
                continue
            preds_for_key[model_name] = pred_dict[key].rename(model_name)

        # Need at least RW + one other model with forecasts
        if "RW" not in preds_for_key or len(preds_for_key) < 2:
            continue

        # Compare each non-RW model against RW
        for model_name, pred_series in preds_for_key.items():
            if model_name == "RW":
                continue

            # Pairwise alignment: actual, RW, and the other model
            df_pair = pd.concat(
                [actual,
                 preds_for_key["RW"],
                 pred_series],
                axis=1,
                join="inner"
            ).dropna()

            if df_pair.empty:
                continue

            # Forecast errors: e = actual - prediction
            err_rw    = df_pair["actual"] - df_pair["RW"]
            err_other = df_pair["actual"] - df_pair[model_name]

            # Diebold–Mariano test (RW as Model 1, other model as Model 2)
            dm_stat, p_val = dm_test(err_rw.values, err_other.values, h=h)

            dm_results.append({
                "Maturity": maturity,
                "Horizon":  h,
                "Model_1":  "RW",
                "Model_2":  model_name,
                "DM_stat":  dm_stat,
                "p_value":  p_val,
                "N":        len(df_pair)
            })

dm_results_df = pd.DataFrame(dm_results)
dm_results_df = dm_results_df.sort_values(["Maturity", "Horizon", "Model_2"])

print(dm_results_df)


In [ ]:
dm_results_df